In [1]:
%%writefile C:\Users\LOKI\churn_app.py
import streamlit as st
import pandas as pd
import joblib
import os

st.title("Customer Churn Prediction")
st.write("Predict whether a telecom customer may churn.")

MODEL_PATH = r"C:\Users\LOKI\churn_model.joblib"

if not os.path.exists(MODEL_PATH):
    st.error("Model file not found. Save your trained model first.")
    st.stop()

model = joblib.load(MODEL_PATH)

st.info("Your churn model is loaded successfully.")

Writing C:\Users\LOKI\churn_app.py


In [2]:
import os

print(os.path.exists(r"C:\Users\LOKI\churn_model.joblib"))

True


In [3]:
import joblib

joblib.dump(
    trained_models["Decision Tree"],
    r"C:\Users\LOKI\churn_model.joblib"
)

NameError: name 'trained_models' is not defined

In [4]:
print("trained_models" in globals())
print("models" in globals())
print("X_train" in globals())
print("y_train" in globals())

False
False
False
False


In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
import joblib

best_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=6,
        class_weight="balanced",
        random_state=42
    ))
])

best_model.fit(X_train, y_train)

joblib.dump(
    best_model,
    r"C:\Users\LOKI\churn_model.joblib"
)

print("Decision Tree trained and saved successfully!")

NameError: name 'preprocessor' is not defined

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
import joblib

# Identify numeric and categorical columns
numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

# Process numerical columns
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Process categorical columns
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Preprocessor created successfully!")

NameError: name 'X_train' is not defined

In [7]:
import pandas as pd

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [8]:
# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

# Remove duplicate rows
df = df.drop_duplicates().copy()

# Separate input features and target
X = df.drop(
    columns=["Churn", "customerID"],
    errors="ignore"
)

y = df["Churn"].str.strip().map({
    "No": 0,
    "Yes": 1
})

# Keep records with a valid target
valid = y.notna()
X = X.loc[valid].copy()
y = y.loc[valid].astype(int)

print("Features:", X.shape)
print("Target:", y.shape)

Features: (7043, 19)
Target: (7043,)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 5634
Testing rows: 1409


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

# Identify numeric and categorical columns
numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

# Preprocessing pipelines
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Create and train the model
best_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=6,
        class_weight="balanced",
        random_state=42
    ))
])

best_model.fit(X_train, y_train)

print("Decision Tree trained successfully!")

Decision Tree trained successfully!


In [11]:
y_pred = best_model.predict(X_test)

print("Classification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["No Churn", "Churn"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

joblib.dump(
    best_model,
    r"C:\Users\LOKI\churn_model.joblib"
)

print("Model saved successfully!")

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.91      0.72      0.80      1035
       Churn       0.51      0.79      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.80      0.74      0.75      1409

Confusion Matrix:
[[748 287]
 [ 78 296]]
Model saved successfully!


In [12]:
import os

model_path = r"C:\Users\LOKI\churn_model.joblib"

print("Model saved:", os.path.exists(model_path))

Model saved: True


In [13]:
%%writefile C:\Users\LOKI\churn_app.py
import streamlit as st
import pandas as pd
import joblib
import os

st.set_page_config(page_title="Customer Churn Prediction")

st.title("Customer Churn Prediction")
st.write("Enter customer details to estimate churn risk.")

model_path = r"C:\Users\LOKI\churn_model.joblib"

if not os.path.exists(model_path):
    st.error("Model file not found. Train and save the model first.")
    st.stop()

model = joblib.load(model_path)

with st.form("customer_form"):
    tenure = st.slider("Tenure (months)", 0, 72, 12)
    monthly_charges = st.number_input(
        "Monthly Charges", min_value=0.0, value=70.0
    )
    total_charges = st.number_input(
        "Total Charges", min_value=0.0, value=840.0
    )
    contract = st.selectbox(
        "Contract", ["Month-to-month", "One year", "Two year"]
    )
    internet_service = st.selectbox(
        "Internet Service", ["DSL", "Fiber optic", "No"]
    )

    submitted = st.form_submit_button("Predict Churn")

if submitted:
    st.warning(
        "This form currently contains only a few features. "
        "The trained model expects the complete set of input columns. "
        "Use the complete-feature form before making a prediction."
    )

Overwriting C:\Users\LOKI\churn_app.py


In [14]:
%pip install streamlit

Note: you may need to restart the kernel to use updated packages.


In [ ]:
!python -m streamlit run "C:\Users\LOKI\churn_app.py"

In [ ]:
%%writefile C:\Users\LOKI\churn_app.py
import streamlit as st
import pandas as pd
import joblib
import os

st.set_page_config(
    page_title="Customer Churn Prediction",
    page_icon="📉",
    layout="wide"
)

st.title("📉 Customer Churn Prediction")
st.write("Enter the customer's details to estimate churn risk.")

MODEL_PATH = r"C:\Users\LOKI\churn_model.joblib"

if not os.path.exists(MODEL_PATH):
    st.error("Model file not found. Please train and save the model first.")
    st.stop()

model = joblib.load(MODEL_PATH)

with st.form("customer_form"):
    st.subheader("Customer Information")

    col1, col2, col3 = st.columns(3)

    with col1:
        gender = st.selectbox("Gender", ["Female", "Male"])
        senior = st.selectbox("Senior Citizen", [0, 1])
        partner = st.selectbox("Partner", ["Yes", "No"])
        dependents = st.selectbox("Dependents", ["Yes", "No"])
        tenure = st.slider("Tenure (months)", 0, 72, 12)
        phone = st.selectbox("Phone Service", ["Yes", "No"])
        multiple_lines = st.selectbox(
            "Multiple Lines",
            ["No", "Yes", "No phone service"]
        )

    with col2:
        internet = st.selectbox(
            "Internet Service", ["DSL", "Fiber optic", "No"]
        )
        security = st.selectbox(
            "Online Security",
            ["No", "Yes", "No internet service"]
        )
        backup = st.selectbox(
            "Online Backup",
            ["No", "Yes", "No internet service"]
        )
        protection = st.selectbox(
            "Device Protection",
            ["No", "Yes", "No internet service"]
        )
        support = st.selectbox(
            "Tech Support",
            ["No", "Yes", "No internet service"]
        )
        tv = st.selectbox(
            "Streaming TV",
            ["No", "Yes", "No internet service"]
        )
        movies = st.selectbox(
            "Streaming Movies",
            ["No", "Yes", "No internet service"]
        )

    with col3:
        contract = st.selectbox(
            "Contract",
            ["Month-to-month", "One year", "Two year"]
        )
        paperless = st.selectbox("Paperless Billing", ["Yes", "No"])
        payment = st.selectbox(
            "Payment Method",
            [
                "Electronic check",
                "Mailed check",
                "Bank transfer (automatic)",
                "Credit card (automatic)"
            ]
        )
        monthly = st.number_input(
            "Monthly Charges", min_value=0.0, value=70.0
        )
        total = st.number_input(
            "Total Charges", min_value=0.0, value=840.0
        )

    submitted = st.form_submit_button("Predict Churn")

if submitted:
    customer = pd.DataFrame([{
        "gender": gender,
        "SeniorCitizen": senior,
        "Partner": partner,
        "Dependents": dependents,
        "tenure": tenure,
        "PhoneService": phone,
        "MultipleLines": multiple_lines,
        "InternetService": internet,
        "OnlineSecurity": security,
        "OnlineBackup": backup,
        "DeviceProtection": protection,
        "TechSupport": support,
        "StreamingTV": tv,
        "StreamingMovies": movies,
        "Contract": contract,
        "PaperlessBilling": paperless,
        "PaymentMethod": payment,
        "MonthlyCharges": monthly,
        "TotalCharges": total
    }])

    try:
        prediction = int(model.predict(customer)[0])

        if hasattr(model, "predict_proba"):
            probability = float(model.predict_proba(customer)[0][1])
        else:
            probability = None

        if prediction == 1:
            st.warning("Prediction: Customer may churn.")
        else:
            st.success("Prediction: Customer may stay.")

        if probability is not None:
            st.metric(
                "Estimated churn probability",
                f"{probability:.1%}"
            )

        st.caption(
            "This is a model estimate, not a guarantee. "
            "The model was trained on a sample dataset."
        )

    except Exception as error:
        st.error(f"Prediction failed: {error}")
        st.info(
            "Check that the model was trained using the same "
            "dataset columns and preprocessing pipeline."
        )

In [ ]:
!python -m streamlit run "C:\Users\LOKI\churn_app.py"

In [ ]:
import os

folder = r"C:\Users\LOKI"

files = [
    "Customer_Churn_Prediction.ipynb",
    "churn_app.py",
    "churn_model.joblib",
    "model_comparison.csv",
    "requirements.txt",
    "README.md"
]


for file in files:
    path = os.path.join(folder, file)
    status = "FOUND" if os.path.exists(path) else "NOT FOUND"
    print(f"{file}: {status}")